# Customer Analytics Mart — Gold Layer

**Medallion Layer**: Gold (`workspace.ecompulse_gold.customer_analytics`)  
**Inputs**:
- `workspace.ecompulse_silver.customers`
- `workspace.ecompulse_silver.orders`
- `workspace.ecompulse_silver.payments`

**Objectives**:
1. Pre-aggregate payments to order grain to prevent Cartesian join fan-out.
2. Sequence customer orders using `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp ASC)`.
3. Aggregate customer order history to derive Customer Lifetime Value (CLV / total spend), order counts, tenure in days, and average basket size.
4. Rank customer spending globally and within state using `DENSE_RANK()`.
5. Segment customers into spending deciles and quartiles using `NTILE(10)` and `NTILE(4)`.
6. Materialize the analytical dataset into Delta Lake table `workspace.ecompulse_gold.customer_analytics`.
7. Provide terminal validation via Databricks `%sql` preview cells.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
customers = spark.read.table("workspace.ecompulse_silver.customers")
orders = spark.read.table("workspace.ecompulse_silver.orders")
payments = spark.read.table("workspace.ecompulse_silver.payments")

In [0]:
# Pre-aggregate payments to order_id grain to prevent fan-out
order_payments_agg = (
    payments
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("order_total_spend"),
        F.count("payment_sequential").alias("payment_installments_count"),
        F.max("payment_installments").alias("max_installments")
    )
)

In [0]:
# Join orders with pre-aggregated payments and apply order sequencing via ROW_NUMBER()
orders_with_payments = (
    orders
    .join(order_payments_agg, on="order_id", how="left")
    .fillna({"order_total_spend": 0.0, "payment_installments_count": 0, "max_installments": 0})
)

order_seq_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("order_purchase_timestamp").asc())
)

orders_sequenced = orders_with_payments.withColumn(
    "order_sequence_num",
    F.row_number().over(order_seq_window)
)

In [0]:
# Aggregate customer orders to derive lifetime metrics and join geographic dimensions
customer_orders_agg = (
    orders_sequenced
    .groupBy("customer_id")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("order_total_spend"), 2).alias("total_spend"),
        F.round(F.avg("order_total_spend"), 2).alias("avg_basket_size"),
        F.min("order_purchase_timestamp").alias("first_purchase_timestamp"),
        F.max("order_purchase_timestamp").alias("last_purchase_timestamp"),
        F.to_date(F.min("order_purchase_timestamp")).alias("first_purchase_date"),
        F.to_date(F.max("order_purchase_timestamp")).alias("last_purchase_date"),
        F.max("order_sequence_num").alias("order_sequence_num"),
        F.datediff(
            F.max("order_purchase_timestamp"),
            F.min("order_purchase_timestamp")
        ).alias("customer_tenure_days")
    )
)

customer_base = (
    customers
    .join(customer_orders_agg, on="customer_id", how="left")
    .fillna({
        "total_orders": 0,
        "total_spend": 0.0,
        "avg_basket_size": 0.0,
        "customer_tenure_days": 0,
        "order_sequence_num": 0
    })
    .withColumn("customer_lifetime_value", F.col("total_spend"))
    .withColumn("avg_order_value", F.col("avg_basket_size"))
)

In [0]:
# Compute spending ranks and RFM decile/quartile segmentation
spend_rank_window = Window.orderBy(F.col("total_spend").desc())
state_spend_rank_window = Window.partitionBy("customer_state").orderBy(F.col("total_spend").desc())
spend_ntile_window = Window.orderBy(F.col("total_spend").desc())

customer_analytics = (
    customer_base
    .withColumn("spend_rank", F.dense_rank().over(spend_rank_window))
    .withColumn("state_spend_rank", F.dense_rank().over(state_spend_rank_window))
    .withColumn("spend_decile", F.ntile(10).over(spend_ntile_window))
    .withColumn("spend_quartile", F.ntile(4).over(spend_ntile_window))
    .withColumn(
        "customer_segment",
        F.when(F.col("spend_decile") == 1, "VIP")
         .when(F.col("spend_decile").between(2, 3), "High Value")
         .when(F.col("spend_decile").between(4, 7), "Medium Value")
         .otherwise("Low Value")
    )
)

In [0]:
# Save customer analytics to Delta Lake table
customer_analytics.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.ecompulse_gold.customer_analytics")

In [0]:
%sql
SELECT 
    customer_id,
    customer_city,
    customer_state,
    total_orders,
    total_spend,
    avg_basket_size,
    customer_tenure_days,
    spend_rank,
    spend_decile,
    customer_segment
FROM workspace.ecompulse_gold.customer_analytics
ORDER BY total_spend DESC
LIMIT 10;